# Tome C1, Chapter 6: Matchings and Flows (Python companion)

Exercises first, solutions at the end. Every theorem of the chapter comes with a certificate, and the code below always returns both halves: a matching *and* a cover of the same size, a flow *and* a cut of the same capacity, a chain partition *and* an antichain. We check them against networkx and scipy, decompose doubly stochastic matrices, test Tutte's condition on every small graph, and run thousands of stable marriages.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools, random, math, time
from collections import defaultdict, deque, Counter
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from scipy.optimize import linprog, linear_sum_assignment
random.seed(6); rng = np.random.default_rng(6)

## Exercises

**P6.1 (★★) Matching with certificates.** Implement bipartite matching by augmenting paths (Theorem 6.1). Return a maximum matching together with a vertex cover of the same size (Theorem 6.4) and, when $X$ cannot be covered, a set $S$ with $|N(S)|<|S|$ (Theorem 6.2). Check everything on random graphs against networkx.

**P6.2 (★★) Dilworth by matching.** Use P6.1 to compute a minimum chain partition and a maximum antichain of a finite poset (the second proof of Dilworth's Theorem 2.14). Try the divisors of $1..n$ and random points of the plane under the product order, where the answer is the longest decreasing sequence.

**P6.3 (★★) Max-flow min-cut.** Implement the augmenting path algorithm with shortest paths (Edmonds–Karp). Return the flow and the minimum cut, check them against networkx, decompose a unit flow into paths (Lemma 6.7), and solve the baseball and project-selection problems of Exercises 6.6 and 6.7.

**P6.4 (★★) Birkhoff.** Make random doubly stochastic matrices (Sinkhorn scaling), decompose them into permutation matrices as in the proof of Theorem 6.9, and count the terms. Then solve assignment problems as linear programs and observe that the optimum is a permutation matrix.

**P6.5 (★★) Tutte's condition.** Check Tutte's theorem (Theorem 6.10) on all graphs with at most $6$ vertices and on random larger ones, by comparing a brute-force search for a violating set $S$ with a maximum matching. Check Petersen's theorem on random cubic graphs and the graph of Figure 6.3.

**P6.6 (★★★) Stable marriages.** Implement deferred acceptance. For small $n$, list all stable matchings by brute force and check Theorems 6.12 and 6.13 and the lattice property (Exercise 6.11). For large $n$ and random preferences, measure the number of proposals and the average ranks of both sides. Replay the lie of Exercise 6.12 and the roommates of Exercise 6.10.

## Solutions

### P6.1
**How to find it.** From each free vertex of $X$, search alternating paths: from $x$ go to any neighbour $y$; if $y$ is free we have an augmenting path, otherwise continue from its partner. If the search from $x$ fails, the vertices it reached give the Hall violator $S$ (second proof of Hall's theorem). After the last augmentation, the vertices reachable from all free vertices of $X$ give König's cover $K=(X\setminus Z)\cup(Y\cap Z)$.

In [ ]:
def bipartite_matching(X, Y, adj):
    Xset = set(X); mate = {}                                  # mate[x] = y and mate[y] = x
    def reach(sources):                        # alternating search; returns (parent map, free y found or None)
        parent = {x: None for x in sources}; q = deque(sources)
        while q:
            x = q.popleft()
            for y in adj[x]:
                if y in parent: continue
                parent[y] = x
                if y not in mate: return parent, y
                parent[mate[y]] = y; q.append(mate[y])
        return parent, None
    violator = None
    for x0 in X:
        if x0 in mate: continue
        parent, y = reach([x0])
        if y is None:
            if violator is None: violator = {v for v in parent if v in Xset}   # |N(S)| = |S| - 1
            continue
        while y is not None:                   # augment along the path back to x0
            x = parent[y]; nxt = mate.get(x)
            mate[x], mate[y] = y, x; y = nxt
    free = [x for x in X if x not in mate]
    Z = set(reach(free)[0]) if free else set()
    cover = {x for x in X if x not in Z} | {y for y in Y if y in Z}
    M = {(x, mate[x]) for x in X if x in mate}
    return M, cover, violator

def random_bipartite(nx_, ny, p):
    X = [('x', i) for i in range(nx_)]; Y = [('y', j) for j in range(ny)]
    adj = {v: set() for v in X + Y}
    for x in X:
        for y in Y:
            if random.random() < p: adj[x].add(y); adj[y].add(x)
    return X, Y, adj

stats = Counter()
for trial in range(2000):
    X, Y, adj = random_bipartite(random.randint(1, 12), random.randint(1, 12), random.choice([0.1, 0.2, 0.4]))
    M, K, S = bipartite_matching(X, Y, adj)
    assert all(y in adj[x] for x, y in M) and len({y for _, y in M}) == len(M)            # a matching
    assert all(u in K or v in K for u in adj for v in adj[u]) and len(K) == len(M)         # a cover of the same size
    G = nx.Graph(); G.add_nodes_from(X + Y); G.add_edges_from((u, v) for u in X for v in adj[u])
    assert len(nx.bipartite.maximum_matching(G, top_nodes=X)) // 2 == len(M)
    if len(M) < len(X):
        N = set().union(*(adj[x] for x in S))
        assert len(N) < len(S); stats['Hall violator'] += 1
    else:
        stats['X covered'] += 1
print(dict(stats), '- matchings, covers and violators all certified')

### P6.2
**How to find it.** The bipartite graph has $x^-y^+$ for $x<y$; a maximum matching gives successor links, hence chains; König's cover gives the antichain of elements $x$ with neither copy in the cover.

In [ ]:
def dilworth(P, less):
    X = [('-', p) for p in P]; Y = [('+', p) for p in P]
    adj = {v: set() for v in X + Y}
    for a in P:
        for b in P:
            if less(a, b): adj[('-', a)].add(('+', b)); adj[('+', b)].add(('-', a))
    M, K, _ = bipartite_matching(X, Y, adj)
    succ = {x[1]: y[1] for x, y in M}; has_pred = set(succ.values())
    chains = []
    for p in P:
        if p not in has_pred:
            c = [p]
            while c[-1] in succ: c.append(succ[c[-1]])
            chains.append(c)
    anti = [p for p in P if ('-', p) not in K and ('+', p) not in K]
    assert sum(map(len, chains)) == len(P) and all(less(a, b) for c in chains for a, b in zip(c, c[1:]))
    assert all(not less(a, b) for a in anti for b in anti) and len(anti) == len(chains)
    return chains, anti

for n in [10, 30, 100]:
    ch, an = dilworth(list(range(1, n + 1)), lambda a, b: a != b and b % a == 0)
    print(f'divisibility on 1..{n}: {len(ch)} chains, antichain {sorted(an)[:12]}{"..." if len(an) > 12 else ""}')
pts = [tuple(rng.random(2)) for _ in range(300)]
ch, an = dilworth(pts, lambda a, b: a != b and a[0] <= b[0] and a[1] <= b[1])
print(f'300 random points: width {len(an)}  (compare 2*sqrt(300) = {2 * math.sqrt(300):.1f}, Erdos-Szekeres / Ulam)')

For the divisors the antichain is $\{\lfloor n/2\rfloor+1,\dots,n\}$ or another set of the same size: no number divides another in it, and the chains $m,2m,4m,\dots$ for odd $m$ show it is optimal. For random points the width is the longest decreasing subsequence, about $2\sqrt n$ (Vershik–Kerov, Logan–Shepp).

### P6.3
**How to find it.** Keep residual capacities on a dictionary of arcs, including reverse arcs; breadth-first search finds a shortest augmenting path; at the end, the vertices reached by the last (failed) search form the minimum cut.

In [ ]:
def max_flow(n_arcs, s, t):
    # n_arcs: dict (u, v) -> capacity. Returns the value, a minimum cut S and the number of augmentations.
    res = defaultdict(float); nbr = defaultdict(set)
    for (u, v), c in n_arcs.items():
        res[(u, v)] += c; nbr[u].add(v); nbr[v].add(u)
    value = 0; augmentations = 0
    while True:
        prev = {s: None}; q = deque([s])
        while q and t not in prev:
            u = q.popleft()
            for w in nbr[u]:
                if w not in prev and res[(u, w)] > 1e-12: prev[w] = u; q.append(w)
        if t not in prev: break
        path = []; v = t
        while prev[v] is not None: path.append((prev[v], v)); v = prev[v]
        d = min(res[a] for a in path)
        for u, v in path: res[(u, v)] -= d; res[(v, u)] += d
        value += d; augmentations += 1
    return value, set(prev), augmentations      # the last search reached exactly the minimum cut

fig = {('s','a'):10, ('s','c'):10, ('a','b'):4, ('a','c'):2, ('a','d'):8, ('c','d'):9, ('d','b'):6, ('b','t'):10, ('d','t'):10}
val, S, k = max_flow(fig, 's', 't')
print('Figure 6.2: max flow', val, 'cut', sorted(S), 'capacity', sum(c for (u, v), c in fig.items() if u in S and v not in S), f'({k} augmentations)')

for trial in range(300):
    n = random.randint(3, 15); arcs = {}
    for u in range(n):
        for v in range(n):
            if u != v and random.random() < 0.3: arcs[(u, v)] = random.choice([random.randint(0, 9), round(random.random() * 5, 3)])
    val, S, _ = max_flow(arcs, 0, n - 1)
    capS = sum(c for (u, v), c in arcs.items() if u in S and v not in S)
    G = nx.DiGraph(); G.add_nodes_from(range(n)); G.add_edges_from((u, v, {'capacity': c}) for (u, v), c in arcs.items())
    assert abs(val - capS) < 1e-9 and abs(val - nx.maximum_flow_value(G, 0, n - 1)) < 1e-9
print('300 random networks (integer and real capacities): flow value = cut capacity = networkx')

The code also shows the integrality theorem at work: on integer capacities all augmentations are by integers. Now Lemma 6.7, reading arc-disjoint paths out of a unit flow, and the two applications.

In [ ]:
def disjoint_paths(arcs_list, s, t):
    arcs = {a: 1 for a in arcs_list}
    f = {a: 0 for a in arcs}                         # an integral flow, built by augmenting paths
    def augment():
        prev = {s: None}; q = deque([s])
        while q:
            u = q.popleft()
            for (a, b) in arcs:
                if a == u and f[(a, b)] == 0 and b not in prev: prev[b] = ((a, b), +1); q.append(b)
                if b == u and f[(a, b)] == 1 and a not in prev: prev[a] = ((a, b), -1); q.append(a)
            if t in prev: break
        if t not in prev: return False
        v = t
        while prev[v] is not None:
            (a, b), sgn = prev[v]; f[(a, b)] += sgn; v = a if sgn == 1 else b
        return True
    val = 0
    while augment(): val += 1
    H = Counter(a for a in arcs if f[a] == 1); paths = []
    for _ in range(int(val)):
        walk = [s]; used = []
        while walk[-1] != t:
            a = next(a for a in H if H[a] and a[0] == walk[-1]); H[a] -= 1; used.append(a); walk.append(a[1])
        path = []                                   # extract a path from the walk (Lemma 5.1)
        for v in walk:
            if v in path: path = path[:path.index(v)]
            path.append(v)
        for a in used: H[a] += 1
        for a in zip(path, path[1:]): H[a] -= 1
        paths.append(path)
    return int(val), paths

D = nx.gnp_random_graph(12, 0.35, seed=3, directed=True)
k, paths = disjoint_paths(list(D.edges()), 0, 11)
print(k, 'arc-disjoint paths from 0 to 11:', paths)
assert len({a for p in paths for a in zip(p, p[1:])}) == sum(len(p) - 1 for p in paths)

In [ ]:
def baseball(wins, left, games, k):
    teams = range(len(wins)); W = wins[k] + left[k]
    if any(wins[i] > W for i in teams): return False
    arcs = {}
    for i, j in itertools.combinations([i for i in teams if i != k], 2):
        g = games[i][j]
        if g: arcs[('s', (i, j))] = g; arcs[((i, j), i)] = g; arcs[((i, j), j)] = g
    for i in teams:
        if i != k: arcs[(i, 't')] = W - wins[i]
    val, _, _ = max_flow(arcs, 's', 't')
    return val == sum(c for (u, v), c in arcs.items() if u == 's')

names = ['Atlanta', 'Philadelphia', 'New York', 'Montreal']
wins = [83, 80, 78, 77]; left = [8, 3, 6, 3]
games = [[0, 1, 6, 1], [1, 0, 0, 2], [6, 0, 0, 0], [1, 2, 0, 0]]
for k in range(4):
    print(f'{names[k]:12s} wins {wins[k]}, {left[k]} left: ', 'can still finish first' if baseball(wins, left, games, k) else 'eliminated')

def brute(wins, games, k):
    pairs = [(i, j) for i, j in itertools.combinations(range(len(wins)), 2) for _ in range(games[i][j])]
    for out in itertools.product([0, 1], repeat=len(pairs)):
        w = list(wins)
        for (i, j), o in zip(pairs, out): w[i if o else j] += 1
        if w[k] >= max(w): return True
    return False
print('brute force agrees:', all(brute(wins, games, k) == baseball(wins, left, games, k) for k in range(4)))

Philadelphia can reach $83$ wins, which ties Atlanta only if Atlanta loses all its remaining games, including the six against New York; but then New York reaches $84$. The flow finds such arguments by itself, for any number of teams. Next, project selection.

In [ ]:
def best_projects(profit, requires):
    m = len(profit); INF = 1 + sum(abs(p) for p in profit); arcs = {}
    for i, p in enumerate(profit):
        if p > 0: arcs[('s', i)] = p
        if p < 0: arcs[(i, 't')] = -p
    for i, j in requires: arcs[(i, j)] = INF
    val, S, _ = max_flow(arcs, 's', 't')
    return sum(p for p in profit if p > 0) - val, sorted(v for v in S if v != 's')

for trial in range(200):
    m = random.randint(1, 10); profit = [random.randint(-9, 9) for _ in range(m)]
    req = [(i, j) for i in range(m) for j in range(m) if i != j and random.random() < 0.15]
    best, chosen = best_projects(profit, req)
    ok = lambda C: all(j in C for i, j in req if i in C)
    bf = max(sum(profit[i] for i in C) for r in range(m + 1) for C in map(set, itertools.combinations(range(m), r)) if ok(C))
    assert best == bf and ok(set(chosen)) and sum(profit[i] for i in chosen) == best
print('project selection: min cut = brute force on 200 random instances')

### P6.4
**How to find it.** Sinkhorn: alternately divide rows and columns by their sums; it converges to a doubly stochastic matrix for positive input. For the decomposition, repeat: find a perfect matching on the positive entries (Hall guarantees it), subtract $\lambda$ times its permutation matrix. Floating point needs a threshold for zero.

In [ ]:
from fractions import Fraction
def random_doubly_stochastic(n, m):
    # an exact rational doubly stochastic matrix: a random weighted sum of m random permutation matrices, normalised
    A = [[Fraction(0)] * n for _ in range(n)]; w = [random.randint(1, 20) for _ in range(m)]
    for wk in w:
        s = random.sample(range(n), n)
        for i in range(n): A[i][s[i]] += Fraction(wk, sum(w))
    return A

def birkhoff(A):
    A = [row[:] for row in A]; n = len(A); terms = []
    while any(x for row in A for x in row):
        X = [('r', i) for i in range(n)]; Y = [('c', j) for j in range(n)]
        adj = {v: set() for v in X + Y}
        for i in range(n):
            for j in range(n):
                if A[i][j] > 0: adj[('r', i)].add(('c', j)); adj[('c', j)].add(('r', i))
        M, _, _ = bipartite_matching(X, Y, adj)
        assert len(M) == n                      # Hall's theorem at work
        sigma = [None] * n
        for (_, i), (_, j) in M: sigma[i] = j
        lam = min(A[i][sigma[i]] for i in range(n))
        for i in range(n): A[i][sigma[i]] -= lam
        terms.append((lam, sigma))
    return terms

for n, m in [(3, 4), (5, 10), (8, 30), (12, 100)]:
    A = random_doubly_stochastic(n, m)
    assert all(sum(r) == 1 for r in A) and all(sum(A[i][j] for i in range(n)) == 1 for j in range(n))
    terms = birkhoff(A)
    R = [[sum(l for l, s in terms if s[i] == j) for j in range(n)] for i in range(n)]
    p = sum(1 for r in A for x in r if x)
    print(f'n={n:2d}: {p:3d} positive entries -> {len(terms):3d} permutation matrices (bound p-n+1 = {p - n + 1}, '
          f'(n-1)^2+1 = {(n - 1) ** 2 + 1}); exact reconstruction: {R == A}')

for n in [4, 7, 10]:
    for trial in range(20):
        Cst = rng.integers(0, 50, (n, n)).astype(float)
        A_eq = []; b_eq = []
        for i in range(n):
            r = np.zeros((n, n)); r[i, :] = 1; A_eq.append(r.ravel()); b_eq.append(1)
            c = np.zeros((n, n)); c[:, i] = 1; A_eq.append(c.ravel()); b_eq.append(1)
        lp = linprog(Cst.ravel(), A_eq=np.array(A_eq), b_eq=b_eq, bounds=(0, None), method='highs-ds')
        rows, cols = linear_sum_assignment(Cst)
        assert abs(lp.fun - Cst[rows, cols].sum()) < 1e-7
        assert np.allclose(np.round(lp.x), lp.x, atol=1e-7)          # simplex returns a vertex: a permutation
    print(f'n={n}: LP relaxation value = best permutation, and the LP vertex is a permutation matrix (20 trials)')

### P6.5
**How to find it.** For every graph and every $S$, count odd components of $G-S$; compare "some $S$ violates" with "the maximum matching (networkx, Edmonds' blossom algorithm) is not perfect". All graphs on $6$ vertices are $2^{15}$; we use bitmasks to keep it fast.

In [ ]:
def odd_components(n, nb, removed):
    seen = removed; odd = 0
    for v in range(n):
        if not seen >> v & 1:
            comp = 0; stack = [v]; seen |= 1 << v
            while stack:
                u = stack.pop(); comp += 1
                free = nb[u] & ~seen
                while free:
                    w = (free & -free).bit_length() - 1; free &= free - 1
                    seen |= 1 << w; stack.append(w)
            odd += comp & 1
    return odd

def tutte_ok(n, nb):
    return all(odd_components(n, nb, S) <= bin(S).count('1') for S in range(1 << n))

def has_pm(n, edges):
    G = nx.Graph(); G.add_nodes_from(range(n)); G.add_edges_from(edges)
    return len(nx.max_weight_matching(G, maxcardinality=True)) * 2 == n

t0 = time.time(); count = Counter()
for n in range(1, 7):
    pairs = list(itertools.combinations(range(n), 2))
    for mask in range(1 << len(pairs)):
        edges = [pairs[i] for i in range(len(pairs)) if mask >> i & 1]
        nb = [0] * n
        for u, v in edges: nb[u] |= 1 << v; nb[v] |= 1 << u
        a, b = tutte_ok(n, nb), has_pm(n, edges)
        assert a == b
        count[(n, a)] += 1
print('all labelled graphs with n <= 6 vertices: Tutte condition <=> perfect matching', f'({time.time() - t0:.1f} s)')
print({n: count[(n, True)] for n in range(1, 7)}, 'graphs with a perfect matching')

In [ ]:
cnt = Counter()
for trial in range(300):
    n = random.choice([10, 20, 30, 50]); G = nx.random_regular_graph(3, n, seed=random.randrange(10 ** 9))
    bridges = len(list(nx.bridges(G))); pm = len(nx.max_weight_matching(G, maxcardinality=True)) * 2 == n
    if bridges <= 2: assert pm                                         # Petersen and Errera
    cnt[(bridges, pm)] += 1
print('random cubic graphs (number of bridges, has perfect matching):', dict(cnt))

S = nx.Graph()
for j in range(3):
    w, k1, k2, k3, k4 = [f'{name}{j}' for name in ('w', 'k1', 'k2', 'k3', 'k4')]
    S.add_edges_from([('s', w), (w, k1), (w, k2), (k1, k3), (k1, k4), (k2, k3), (k2, k4), (k3, k4)])
print('Figure 6.3: degrees', set(dict(S.degree()).values()), ', vertices', S.number_of_nodes(), ', bridges', len(list(nx.bridges(S))),
      ', maximum matching', len(nx.max_weight_matching(S, maxcardinality=True)))

Random cubic graphs almost never have bridges, so they almost always have perfect matchings; the $16$-vertex graph, with its three bridges, has a maximum matching of $7$ edges, one short.

### P6.6
**How to find it.** Keep for each proposer the index of its next proposal and for each receiver an inverse ranking table, so that each proposal costs $O(1)$. For brute force, try all $n!$ perfect matchings and test the $n^2$ pairs.

In [ ]:
def gale_shapley(P, R):
    n = len(P); rank = [{p: i for i, p in enumerate(R[r])} for r in range(n)]
    nxt = [0] * n; held = [None] * n; free = list(range(n)); proposals = 0
    while free:
        p = free.pop(); r = P[p][nxt[p]]; nxt[p] += 1; proposals += 1
        if held[r] is None: held[r] = p
        elif rank[r][p] < rank[r][held[r]]: free.append(held[r]); held[r] = p
        else: free.append(p)
    match = [None] * n
    for r, p in enumerate(held): match[p] = r
    return match, proposals

def is_stable(match, P, R):
    n = len(P); inv = {r: p for p, r in enumerate(match)}
    rp = [{r: i for i, r in enumerate(P[p])} for p in range(n)]; rr = [{p: i for i, p in enumerate(R[r])} for r in range(n)]
    return not any(rp[p][r] < rp[p][match[p]] and rr[r][p] < rr[r][inv[r]] for p in range(n) for r in range(n))

def random_prefs(n):
    return [random.sample(range(n), n) for _ in range(n)], [random.sample(range(n), n) for _ in range(n)]

sizes = Counter()
for trial in range(300):
    n = random.randint(2, 7); P, R = random_prefs(n)
    M0, _ = gale_shapley(P, R)
    stable = [list(m) for m in itertools.permutations(range(n)) if is_stable(list(m), P, R)]
    assert M0 in stable
    rp = [{r: i for i, r in enumerate(P[p])} for p in range(n)]; rr = [{p: i for i, p in enumerate(R[r])} for r in range(n)]
    for M in stable:
        assert all(rp[p][M0[p]] <= rp[p][M[p]] for p in range(n))                    # proposer-optimal
        inv0 = {r: p for p, r in enumerate(M0)}; inv = {r: p for p, r in enumerate(M)}
        assert all(rr[r][inv0[r]] >= rr[r][inv[r]] for r in range(n))                # receiver-pessimal
    for M, M2 in itertools.combinations(stable, 2):                                   # lattice: the join is stable
        J = [M[p] if rp[p][M[p]] <= rp[p][M2[p]] else M2[p] for p in range(n)]
        assert sorted(J) == list(range(n)) and J in stable
    sizes[len(stable)] += 1
print('300 random instances, n <= 7: proposer-optimal, receiver-pessimal, lattice: all checked')
print('number of stable matchings found:', dict(sorted(sizes.items())))

In [ ]:
ns = [100, 200, 500, 1000, 2000]; props, prank, rrank = [], [], []
for n in ns:
    a, b, c = [], [], []
    for trial in range(5):
        P, R = random_prefs(n); M, k = gale_shapley(P, R)
        rp = [{r: i for i, r in enumerate(P[p])} for p in range(n)]; rr = [{p: i for i, p in enumerate(R[r])} for r in range(n)]
        a.append(k); b.append(np.mean([rp[p][M[p]] + 1 for p in range(n)])); c.append(np.mean([rr[M[p]][p] + 1 for p in range(n)]))
    props.append(np.mean(a)); prank.append(np.mean(b)); rrank.append(np.mean(c))
    H = sum(1 / i for i in range(1, n + 1))
    print(f'n={n:5d}: proposals {np.mean(a):9.0f} (n H_n = {n * H:8.0f}),  proposer rank {np.mean(b):5.2f} (ln n = {math.log(n):.2f}),'
          f'  receiver rank {np.mean(c):6.1f} (n/ln n = {n / math.log(n):.1f})')
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(ns, prank, 'o-', label='proposers'); ax[0].plot(ns, np.log(ns), '--', label='ln n'); ax[0].set_xscale('log'); ax[0].legend(); ax[0].set_title('average rank of partner')
ax[1].plot(ns, rrank, 's-', color='C1', label='receivers'); ax[1].plot(ns, np.array(ns) / np.log(ns), '--', color='C2', label='n / ln n')
ax[1].set_xscale('log'); ax[1].set_yscale('log'); ax[1].legend(); ax[1].set_title('average rank of partner')
plt.tight_layout(); plt.show()

The convergence is slow and the samples are small, but the asymmetry is unmistakable: the proposers get partners of rank about $\ln n$, the receivers of rank about $n/\ln n$. Proposing is worth a factor of about $n/\ln^2n$. Finally, the lie and the roommates.

In [ ]:
A, B, Cc = 0, 1, 2; x, y, z = 0, 1, 2
P = [[x, y, z], [x, z, y], [y, x, z]]
R_true = [[Cc, A, B], [A, B, Cc], [A, B, Cc]]
R_lie = [[Cc, B, A], [A, B, Cc], [A, B, Cc]]
for name, R in [('truth', R_true), ('x lies', R_lie)]:
    M, _ = gale_shapley(P, R); inv = {r: p for p, r in enumerate(M)}
    print(f'{name:7s}: x gets {"ABC"[inv[x]]}, y gets {"ABC"[inv[y]]}, z gets {"ABC"[inv[z]]}')

pref = {'a': 'bcd', 'b': 'cad', 'c': 'abd', 'd': 'abc'}
for pairing in [(('a', 'b'), ('c', 'd')), (('a', 'c'), ('b', 'd')), (('a', 'd'), ('b', 'c'))]:
    mate = {}
    for u, v in pairing: mate[u], mate[v] = v, u
    blocking = [(u, v) for u, v in itertools.combinations('abcd', 2) if mate[u] != v
                and pref[u].index(v) < pref[u].index(mate[u]) and pref[v].index(u) < pref[v].index(mate[v])]
    print('roommates', pairing, 'blocked by', blocking)